# 05 — Sensibilidade e capacidade adaptativa
- **Sensibilidade (S):** matrículas, faixa etária (bebês e crianças pequenas pesam mais), período integral.
- **Capacidade adaptativa (C):** vegetação no entorno (NDVI do notebook 03; opcional: Cobertura Vegetal 2017 do GeoSampa),
  ar-condicionado já existente (inventário SME, se obtido).
- **Contexto social (opcional):** renda do setor censitário (IBGE) — entra como sensibilidade.

In [ ]:
!pip -q install geopandas

In [ ]:
# --- setup comum (Colab) ---
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/2_PESQUISA/02_PROJETOS-PESQUISA-MODELAGEM/ESCOLAS-CONFORTO-TERMICO-SME'
import os, sys
os.chdir(ROOT); sys.path.append(f'{ROOT}/src')
RAW, INT, PRO, OUT = [f'{ROOT}/{p}' for p in ['data/raw', 'data/interim', 'data/processed', 'outputs']]
for p in [RAW, INT, PRO, OUT]:
    os.makedirs(p, exist_ok=True)
CRS = 'EPSG:31983'          # SIRGAS 2000 / UTM 23S
BUFFERS = [50, 100, 150, 250]
BUFFER_REF = 100            # buffer usado no score (≈ 1 pixel termal nativo do Landsat)

In [ ]:
import requests, glob, numpy as np, pandas as pd, geopandas as gpd
esc = gpd.read_file(f'{INT}/escolas.gpkg', layer='escolas'); esc['cod_esc'] = esc['cod_esc'].astype(str)
temp = pd.read_parquet(f'{PRO}/temperatura_escolas.parquet')

# peso por faixa etária/turno (editável — vai para a aba "parametros" do Excel)
PESO_TIPO = [  # (regex em tipoesc, peso_idade, integral)
    (r'CEI|CR\.?P|CRECHE|CCI|CIPS', 1.00, 1),   # 0–3 anos, período integral
    (r'CEMEI',                     0.90, 1),
    (r'EMEI',                      0.80, 0),   # 4–5 anos
    (r'EMEBS',                     0.70, 0),
    (r'EMEF',                      0.50, 0),   # 6–14 anos (EMEFM idem)
]
t = esc['tipoesc'].str.upper().fillna('')
esc['peso_idade'] = np.nan; esc['integral'] = 0
for rx, p, integ in PESO_TIPO:
    m = t.str.contains(rx) & esc['peso_idade'].isna()
    esc.loc[m, 'peso_idade'] = p; esc.loc[m, 'integral'] = integ
esc['peso_idade'] = esc['peso_idade'].fillna(0.5)
esc[['tipoesc', 'peso_idade', 'integral']].drop_duplicates().sort_values('peso_idade', ascending=False)

In [ ]:
# matrículas — procura os conjuntos da SME no portal e mostra opções
API = 'https://dados.prefeitura.sp.gov.br/api/3/action/package_search'
r = requests.get(API, params={'q': 'matriculas escola', 'fq': 'organization:educacao1', 'rows': 20}, timeout=60).json()
for p in r['result']['results']:
    print(p['name'], '|', p['title'])

In [ ]:
# escolha o dataset/arquivo de matrículas por escola e ajuste as colunas:
MATRIC_CSV = None          # ex.: f'{RAW}/sme/matriculas_por_escola.csv' (baixe pelo id listado acima)
COL_ESC, COL_QTD = 'codesc', None   # COL_QTD=None → conta linhas (microdados: 1 linha por aluno)

if MATRIC_CSV and os.path.exists(MATRIC_CSV):
    mt = pd.read_csv(MATRIC_CSV, sep=None, engine='python', encoding='latin-1', dtype=str)
    mt.columns = mt.columns.str.lower()
    if COL_QTD:
        mt[COL_QTD] = pd.to_numeric(mt[COL_QTD], errors='coerce'); mat = mt.groupby(COL_ESC)[COL_QTD].sum()
    else:
        mat = mt.groupby(COL_ESC).size()
    esc = esc.merge(mat.rename('matriculas'), left_on='cod_esc', right_index=True, how='left')
else:
    esc['matriculas'] = np.nan
    print('⚠️ sem matrículas: o score usará só peso_idade/integral na sensibilidade')

In [ ]:
# capacidade adaptativa: NDVI (já calculado) + cobertura vegetal GeoSampa (opcional) + AC existente (opcional)
vul = esc[['cod_esc', 'peso_idade', 'integral', 'matriculas']].merge(
    temp[['cod_esc', 'NDVI_median']], on='cod_esc', how='left')

cv = glob.glob(f'{RAW}/geosampa/*obertura*egetal*.gpkg')
if cv:
    veg = gpd.read_file(cv[0]).to_crs(CRS)
    buf = gpd.read_file(f'{INT}/escolas.gpkg', layer=f'buffer_{BUFFER_REF}m')
    inter = gpd.overlay(buf, veg[['geometry']], how='intersection')
    inter['a'] = inter.area
    pv = inter.groupby('cod_esc')['a'].sum() / (np.pi * BUFFER_REF ** 2)
    vul = vul.merge(pv.rename('pct_vegetacao').reset_index(), on='cod_esc', how='left').fillna({'pct_vegetacao': 0})

ac = f'{RAW}/sme/inventario_ac.csv'   # colunas: cod_esc, possui_ac (0/1) ou pct_salas_com_ac
if os.path.exists(ac):
    vul = vul.merge(pd.read_csv(ac, dtype={'cod_esc': str}), on='cod_esc', how='left')

renda = glob.glob(f'{RAW}/ibge/setores*.gpkg')   # setores censitários com coluna "renda_media"
if renda:
    st = gpd.read_file(renda[0]).to_crs(CRS)[['renda_media', 'geometry']]
    j = gpd.sjoin(esc[['cod_esc', 'geometry']], st, how='left', predicate='within')
    vul = vul.merge(j[['cod_esc', 'renda_media']].drop_duplicates('cod_esc'), on='cod_esc', how='left')

vul.to_parquet(f'{PRO}/vulnerabilidade_escolas.parquet')
vul.describe(include='all').T